This is a step-by-step instruction to bare install Snipe IT on a Ubuntu Server. You can simply copy it into your Linux Command Line.
Source 1: https://syncbricks.com/snipe-it-instsallation-ubuntu-20-04/
Source 2: https://snipe-it.readme.io/docs
Source 3: Myself

Install LAMP (Linux, Apache, MySQL, PHP) environment

In [ ]:
# Update packages after installation
sudo apt update && sudo apt upgrade

# Install HTTP server
sudo apt install apache2
systemctl start apache2 && systemctl enable apache2
# Check status of the server
systemctl status apache2
apache2 -v
# Open ports on the firewall
sudo ufw allow 80/tcp
sudo ufw allow 443/tcp
sudo ufw reload
sudo a2enmod rewrite
systemctl restart apache2

# Install Database
sudo apt install mysql
sudo systemctl start mysql && sudo systemctl enable mysql
sudo systemctl status mysql
sudo mysql_secure_installation

mysql -u root -p
CREATE DATABASE snipeit;
SHOW databases;
CREATE USER snipe_user@localhost IDENTIFIED BY 'Password';
GRANT ALL PRIVILEGES ON snipeit.* TO snipe_user@localhost;
FLUSH PRIVILEGES;
EXIT;


# Install PHP
sudo apt install php php-cli php-fpm php-json php-common php-mysql php-zip php-gd php-mbstring php-curl php-xml php-pear php-bcmath
sudo apt install php php-bcmath php-bz2 php-intl php-gd php-mbstring php-mysql php-zip php-opcache php-pdo php-calendar php-ctype php-exif php-ffi php-fileinfo php-ftp php-iconv php-intl php-json php-mysqli php-phar php-posix php-readline php-shmop php-sockets php-sysvmsg php-sysvsem php-sysvshm php-tokenizer php-curl php-ldap -y



Install & Config SnipeIT

In [ ]:
# Install the latest version using Git
cd /var/www/
git clone https://github.com/snipe/snipe-it snipe-it

# Grant access
cd /var/www/snipe-it
sudo chown -R www-data:www-data ./storage
sudo chmod -R 755 ./storage

# Open SnipeIT's config file
cd /var/www/snipe-it
sudo cp /var/www/snipe-it/.env.example /var/www/snipe-it/.env
sudo vim /var/www/snipe-it/.env

# Lines needs to be edited
APP_URL=ams.example.com
APP_TIMEZONE='UTC'
DB_CONNECTION=mysql
DB_HOST=localhost
DB_DATABASE=snipeit
DB_USERNAME=snipe_user
DB_PASSWORD=Password
MAIL_MAILER=smtp
MAIL_HOST=smtp.example.com
MAIL_PORT=587
MAIL_USERNAME=YOURUSERNAME
MAIL_PASSWORD=YOURPASSWORD
MAIL_FROM_ADDR=you@example.com
MAIL_FROM_NAME=Snipe-IT
MAIL_REPLYTO_ADDR=you@example.com
MAIL_REPLYTO_NAME=Snipe-IT
MAIL_AUTO_EMBED=true
MAIL_AUTO_EMBED_METHOD=base64
MAIL_TLS_VERIFY_PEER=false


Install PHP Composer

In [ ]:
# Download PHP Composer
cd /var/www/snipe-it
curl -sS https://getcomposer.org/installer | php

# Install method 1: snipeit
php composer.phar update
php composer.phar install --no-dev --prefer-source

# Install method 2: Source 1
mv composer.phar /usr/local/bin/composer 
composer update
composer install --no-dev --prefer-source

# Generate Key
php artisan key:generate

Config apache2 as following: (000-default can be used instead)

In [ ]:
sudo a2dissite 000-default.conf
sudo vim /etc/apache2/sites-available/snipe-it.conf

# Lines needed:
<VirtualHost *:80>
ServerName snipe-it.syncbricks.com
DocumentRoot /var/www/snipe-it/public
<Directory /var/www/snipe-it/public>
Options Indexes FollowSymLinks MultiViews
AllowOverride All
Order allow,deny
allow from all
</Directory>
</VirtualHost>

# Enable edited config for apache2
a2ensite snipe-it.conf

# Restart the apache2 after configuration is done
systemctl restart apache2